<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## ✅ M2 Lab 2 · 参考答案：CustomPlotTool + 插图报告

**一句话**：官方完整解答：自定义画图工具交给 Report Writer，并用 `Solution/tasks.yaml` 要求它把图插进报告。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ Solution</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 C1M2_Lab_2_deep_research_p2_solution.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：C1M2_Lab_2_deep_research_p2.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📂 agents.yaml 用上一级；tasks.yaml 用 Solution/ 自带的</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：在 Lab 1 的可靠研究 crew 上，让最终报告 **自动配图**。
答案由三部分组成：① `CustomPlotTool`（BaseTool 子类）的 `name / description / _run`；② `report_writer` 的 `tools=[CustomPlotTool()]`；③ `Solution/tasks.yaml` 里 `write_final_report` 补充的说明。
概念讲解见练习 notebook，这里讲「答案为什么这样写」和它与练习版的区别。

</div>

<img src="../images/lab2-agents-tasks-diagram.png" width="900">

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📌 和你的练习版对照

</div>

| 填空处 | 参考答案 | 你的练习版 |
|---|---|---|
| 工具 `name` / `description` / `_run` | 同下方代码 | 完全相同 ✅（只是排版不同） |
| `report_writer` 工具 | `tools=[CustomPlotTool()]` | 相同 ✅ |
| tasks.yaml 补充 | 用 `deep_research_crew/plots/` 路径插图 | 原为空，已补成 `plots/` 路径（见下） |
| 运行 | `kickoff()` | `await kickoff_async()` |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**本 notebook 为能直接运行所做的改动**：
1. `sys.path.insert(0, "..")` → import 上一级的 `utils.py`；
2. `config/agents.yaml` → `../config/agents.yaml`；`config/tasks.yaml` → `tasks.yaml`（**Solution/ 自带的已解答版本**，上一级那份是练习用的）；
3. `kickoff()` → `await kickoff_async()`。
报告和图都会落在 `Solution/` 下（`research_report-p2.md`、`plots/`），不覆盖练习版结果。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 1 步：导入 + 环境变量

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔑 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">utils.get_*_api_key()</code></div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">load_dotenv(find_dotenv())</code>：从当前目录往上找 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">.env</code>（找到仓库根的那份），再读环境变量。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🤖 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">os.environ[&quot;MODEL&quot;]</code></div><div style="font-size:0.92em; line-height:1.6;">没给 agent 传 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">llm=</code> 时，CrewAI 用 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">MODEL</code> 环境变量当默认模型（不设则用 crewai 内置默认，1.15.22 是 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">gpt-4.1-mini</code>）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧪 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">CREWAI_TESTING=true</code></div><div style="font-size:0.92em; line-height:1.6;">让 crewai 认为处于测试环境：<b>跳过首次运行时「是否开启 tracing」的交互确认</b>，notebook 不会卡住等输入。</div></div></div></div>

In [1]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
import sys
sys.path.insert(0, "..")   # 🆕 Solution/ 子目录：让上一级的 utils.py 可被 import
os.environ["CREWAI_TESTING"] = "true"   # 跳过首次运行的 tracing 交互确认
from utils import get_openai_api_key, get_exa_api_key
from IPython.display import Markdown
import yaml

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"   # 没指定 llm 的 agent 默认用它
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 2 步：CustomPlotTool —— 答案在类的前三行

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🏷️ name</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">&quot;Create custom plots&quot;</code>：简短、动词开头，模型一看就知道干什么。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">📝 description</div><div style="font-size:0.92em; line-height:1.6;">说了三件事：<b>作用</b>（根据研究结果自动画图）、<b>输入要求</b>（核查过的信息）、<b>怎么传</b>（把目前全部已验证信息作为一个字符串传入）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">⚙️ _run(self, research: str) -&gt; str</div><div style="font-size:0.92em; line-height:1.6;">一个字符串参数 → 自动生成 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">args_schema</code>；返回字符串（图片路径列表）给 agent。</div></div></div></div>

In [2]:
# import packages needed for the custom tool
from crewai.tools import BaseTool
from crewai import LLM
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json

# Define the custom tool for creating plots
class CustomPlotTool(BaseTool):   # 继承 BaseTool = 面向对象写法的自定义工具
    ### START CODE HERE ###
    name: str = "Create custom plots"   # 模型看到的工具名
    description: str = ("This a tool for automatically creating custom plots based on a research result. "   # 模型据此判断何时调用、传什么
                        "This tools automatically generates the plots from a text input, which should have fact checked information. "
                        "Pass the full validated information gathered so far as a string."
                        )
    def _run(self, research: str) -> str:   # 参数签名 → 自动生成 args_schema {research: str}
    ### END CODE HERE ###
        try:   # ① 先用下面的 prompt 让 LLM 从研究文本里抽出「可画图的数据」
            extraction_prompt = f"""
            You are an expert data visualization assistant. Analyze the provided research text and identify meaningful, insightful charts that can be created to visualize quantifiable data supporting the research's key insights and findings. Only suggest charts for data that includes numerical values, measurable trends, comparisons, or categorical distributions that can be effectively plotted.

            Focus on creating visualizations that highlight trends, comparisons, distributions, or relationships that add value to the research. Avoid suggesting charts for purely qualitative or non-quantifiable information.

            For each chart, provide a JSON object with:
              - "chart_type" (string: choose from "line" for trends over time/continuous, "bar" for comparisons, "histogram" for distributions, "scatter" for relationships, "pie" for proportions)
              - "x_axis" (string: variable name for x-axis, e.g., "year", "category")
              - "y_axis" (string: variable name for y-axis, e.g., "value", "count")
              - "color" (string: optional variable for color grouping/hue, or null if not applicable)
              - "Title" (string: descriptive, insightful title that explains what the chart shows)
              - "data" (dictionary: keys matching x_axis, y_axis, and color variables; values as lists of extracted numerical/categorical data from the research)

            Ensure data is accurately extracted and formatted as lists. If a variable has multiple series (e.g., for color), include all in the data dictionary.

            If no quantifiable data suitable for meaningful visualization is present in the research, return an empty array [].

            Text:
            {research}

            Example output (return valid JSON only):
            [
              {{"chart_type": "line", "x_axis": "year", "y_axis": "funding_amount", "color": "sector", "Title": "AI Research Funding Trends by Sector", "data": {{"year": [2020, 2021, 2022], "funding_amount": [2.5, 3.8, 5.2], "sector": ["Healthcare", "Finance", "Tech"]}}}},
              {{"chart_type": "bar", "x_axis": "tool_name", "y_axis": "adoption_rate", "color": null, "Title": "Market Adoption Rates of AI Tools", "data": {{"tool_name": ["ToolA", "ToolB", "ToolC"], "adoption_rate": [45, 67, 23]}}}}
            ]

            Return only the JSON array, no additional text or explanations.
            """
            llm = LLM(model="gpt-4o-mini",)  # Initialize the LLM instance   # 工具内部自己又调了一次 LLM（独立于 agent 的 LLM）
            llm_response = llm.call([{"role": "user", "content": extraction_prompt}])   # 同步调用，返回字符串

            # Clean the response to extract just the JSON part
            llm_response = llm_response.strip()
            if llm_response.startswith('```json'):   # 去掉模型可能包的 ```json 代码块围栏
                llm_response = llm_response[7:]  # Remove ```json
            if llm_response.endswith('```'):
                llm_response = llm_response[:-3]  # Remove ```
            llm_response = llm_response.strip()

            # --- Step 2: Parse the LLM output ---
            charts_data = json.loads(llm_response)   # ② 解析成 list[dict]，每个 dict 描述一张图

            if not isinstance(charts_data, list) or len(charts_data) == 0:
                return "No information found in the research to visualize."

            plots_created = []

            # --- Step 3: Create plots for each chart ---
            for i, chart_info in enumerate(charts_data):
                try:
                    # Extract chart configuration
                    chart_type = chart_info.get("chart_type", None).lower()
                    x_axis = chart_info.get("x_axis", "x")
                    y_axis = chart_info.get("y_axis", "y") 
                    title = chart_info.get("Title", f"Chart {i+1}")
                    hue = chart_info.get("color", None)
                    data = chart_info.get("data", {})

                    # Create DataFrame from the data
                    df = pd.DataFrame(data)   # ③ 每张图的数据 → DataFrame → seaborn 画图

                    if df.empty:
                        continue

                    # Create the plot
                    plt.figure(figsize=(10, 6))

                    if chart_type == "line":
                        sns.lineplot(data=df, x=x_axis, y=y_axis, marker="o", hue=hue)
                    elif chart_type in ["bar", "column"]:
                        sns.barplot(data=df, x=x_axis, y=y_axis, hue=hue)
                    elif chart_type == "histogram":
                        plt.hist(df[y_axis], bins=10, alpha=0.7, hue=hue)   # ⚠️ 课程 bug：plt.hist 没有 hue 参数 → 直方图会失败并被跳过
                        plt.xlabel(y_axis)
                        plt.ylabel("Frequency")
                    elif chart_type == "scatter":
                        # Default to scatter plot
                        sns.scatterplot(data=df, x=x_axis, y=y_axis, hue=hue)
                    elif chart_type == "pie":
                        # For pie chart, assume y_axis is values, x_axis is labels
                        plt.pie(df[y_axis], labels=df[x_axis], autopct='%1.1f%%', startangle=90)
                        plt.title(title)
                        plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.

                    plt.title(title)
                    plt.xticks(rotation=45)
                    plt.tight_layout()

                    # --- Step 4: Save the plot ---
                    os.makedirs("plots", exist_ok=True)   # ④ 存到 cwd 下的 plots/
                    filename = f"plots/plot_{i+1}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
                    plt.savefig(filename, dpi=300, bbox_inches='tight')
                    plt.close()   # 关闭图，避免内存累积

                    plots_created.append(filename)

                except Exception as e:
                    print(f"Error creating chart {i+1}: {str(e)}")
                    continue

            if plots_created:
                return f"Successfully created {len(plots_created)} plots: {', '.join(plots_created)}"   # ⑤ 返回给 agent 的只是一段文字：图的文件路径
            else:
                return "No plots could be created from the extracted data."

        except json.JSONDecodeError as e:   # 工具出错时返回字符串而不是抛异常 → agent 能读到并继续
            return f"Error parsing LLM response as JSON: {str(e)}"
        except Exception as e:
            return f"Error generating smart plot: {str(e)}"

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 3 步：搜索 / 抓取工具 + 4 个 agent

</div>

两个课程视频里没讲、lab 里加上的参数（都是为了跑得快、别撞限流）：
- `max_rpm=150`：该 agent 每分钟最多 150 次 LLM 请求，超了自动等待；
- `max_iter=15`：思考↔调工具 最多 15 轮（默认 25），到上限必须给出「当前最好答案」。

In [3]:
# create tools instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))   # deprecated 别名（新名 ExaSearchTool），仍可用
scrape_website_tool = ScrapeWebsiteTool()

/var/folders/xx/ypnl5f2n0y7b48w_pgxyhqt80000gn/T/ipykernel_36781/3990803208.py:2: DeprecationWarning: EXASearchTool is deprecated and will be removed in a future release; use ExaSearchTool instead.
  exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))   # deprecated 别名（新名 ExaSearchTool），仍可用


In [4]:
# load the configuration file for the agents  (🆕 ../ = lab 根目录)
with open('../config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)


# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )

internet_researcher = Agent( 
        config=agent_config['internet_researcher'],
        verbose=True,
        tools=[exa_search_tool, scrape_website_tool],
        max_rpm=150,
        max_iter=15
        )

fact_checker = Agent(
        config=agent_config['fact_checker'],
        verbose=True,
        tools=[exa_search_tool, scrape_website_tool],
        max_rpm=150,
        max_iter=15
        )

report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        ### START CODE HERE ### 
        # add the automatic plot tool
        tools=[CustomPlotTool()],   # ← 答案：画图工具只给写报告的 agent
        ### END CODE HERE ###
        max_rpm=150,
        max_iter=15
        )

为什么只给 Report Writer？画图需要的是 **已核查** 的数据；writer 拿到的 context 正是 fact checker 的输出。给 researcher 画图会把未验证的数字画进去。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 4 步：guardrail（Lab 1 答案）

</div>

In [5]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw 
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower): 
        return (False, 
                "The report must include a Citations (or References) section with a header like '## Citations'"
                )
    return (True, output)

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 5 步：tasks —— 读 `Solution/tasks.yaml`

</div>

参考答案在 `write_final_report` 的 description 里加了「用自定义工具画图、把图插入报告」，expected_output 里加了「尽可能包含图」：

In [6]:
from pathlib import Path
yaml_text = Path("tasks.yaml").read_text()                   # Solution/ 自带的已解答版本
print(yaml_text[yaml_text.index("write_final_report:"):])

write_final_report:
  # update the description to include the custom tool usage with the
  # reviewed information received from the previous task
  description: >
    Create a comprehensive report that answers the original query "{user_query}"
    using all verified research data. Structure it with clear sections, include citations,
    and provide actionable insights.
    ### START CODE HERE ###
    Use the custom tool to create intuitive charts based on the verified information. 
    Include the generated charts into the final report, any chart should be loadded from the deep_research_crew/plots/ path, so you just append the file name after that'
    ### END CODE HERE ###
  # update the expected output to include charts whenever possible 
  expected_output: >
    A final research report containing: 1) Executive summary, 2) Detailed
    findings that answer the user query, 3) Supporting evidence and analysis, 4) Complete
    source citations, 5) Key insights and recommendations
    ##

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**参考答案的小问题**（不影响运行，影响图片显示）：
- 它让模型从 `deep_research_crew/plots/` 加载图片——那是课程 CLI 项目的目录结构；这里工具把图存在 cwd 下的 `plots/`。本次运行模型照抄了工具返回的 `plots/...` 路径，图正好能显示；但模型若听从说明拼上 `deep_research_crew/`，链接就坏了。练习版的 `config/tasks.yaml` 已直接写成 `plots/` 路径。
- 句末多了一个孤立的 `'`、`loadded` 拼写错误；`### START CODE HERE ###` 标记在 `>` 折叠块里，会作为普通文字进入 prompt。

</div>

In [7]:
# load the configuration file for the tasks  (🆕 Solution/tasks.yaml = 已解答版)
with open('tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)


# create the tasks using the configuration
create_research_plan = Task( 
    config=task_config['create_research_plan'],
    agent=research_planner 
)

gather_research_data = Task(
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task(
    config=task_config['verify_information_quality'],
    agent=fact_checker, 
)

write_final_report = Task( 
    config=task_config['write_final_report'],
    agent=report_writer, 
    guardrails=[write_report_guardrail],   # 报告仍要过结构检查
)

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 6 步：hook + Crew + 输入

</div>

`after_kickoff_callbacks` 里的函数在 **所有 task 结束后** 被调用，参数 `result` 是 `CrewOutput`：
`result.tasks_output` = 每个 task 的 `TaskOutput` 列表，`[-1].raw` 就是最终报告的 Markdown 原文。
（对应还有 `before_kickoff_callbacks`：kickoff 前调用，可以修改 `inputs`。）

In [8]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report-p2.md"   # 存在 Solution/ 下
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

    return result   # 🆕 必须把 result 还回去：hook 的返回值会替换 kickoff 的返回值（原代码没有这行 → result 变成 None）

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 修正课程代码：hook 末尾加了 `return result`**
crewai 源码（1.15.22 `crew.py`）里是 `for cb in self.after_kickoff_callbacks: result = cb(result)` ——每个 after-kickoff 回调的 **返回值会替换** crew 的最终结果（字段说明："It may be used to adjust the output of the crew"）。
课程的 `save_file_hook` 只存文件、不返回 → `kickoff` 返回 `None`，后面再用 `result.raw` 就会 `AttributeError: 'NoneType' object has no attribute ...`（课程 notebook 恰好没再用 `result`，所以没暴露）。

</div>

In [9]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],
    # add memory to the crew
    memory=True,   # 统一 Memory：存于 ~/Library/Application Support/Solution/memory（与 Lab 1 解答共用，因为文件夹名都叫 Solution）
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]
)

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026-10 版本差异：`memory=True` 背后是什么**（已对照 crewai 1.15.22 源码）
- 旧版课程时代：short-term / long-term / entity 三种记忆（ChromaDB + SQLite）。
- 现在：统一的 `Memory` 类（`crewai/memory/unified_memory.py`）——存储默认 **LanceDB**，embedding 默认 OpenAI `text-embedding-3-large`，存盘时还会用 agent 的 LLM 分析内容、打分类和重要度。
- 存在 `~/Library/Application Support/<notebook 所在文件夹名>/memory`（`crewai_core.paths.db_storage_path()` 用 `Path.cwd().name` 当 app 名；设 `CREWAI_STORAGE_DIR` 可改），**跨运行持久**：同一文件夹、同名 crew（默认 scope `/crew/crew`）再次运行会召回上次的记忆。
- 召回点：每个 task 开始前用 task 描述 `recall(limit=5)`，命中的记忆以 "Relevant memories:" 拼进 prompt。
- 还会给每个 agent **自动加两个工具** `Search memory` / `Save to memory`（`crewai/tools/memory_tools.py`），所以 verbose 日志里会看到 agent 自己调用 memory 工具（"Found memories" / "Saving N items to memory"）。

</div>

In [10]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = {
        "user_query": "Evaluate the top five emerging AI tools for automating competitive market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm."   # ✏️ 改这里：研究问题
}
### END CODE HERE ###   

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 7 步：运行

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异：`crew.kickoff()` → `await crew.kickoff_async()`**
原代码 `result = deep_research_crew.kickoff(inputs=inputs)` 在 notebook 里会报
`RuntimeError: Agent execution was invoked synchronously from within a running event loop`。
原因：Jupyter 内核本身一直跑着 asyncio 事件循环，crewai 1.15 的 agent 检测到它就拒绝同步执行。`kickoff_async` 在 1.15.22 里就是 `await asyncio.to_thread(self.kickoff, ...)`，把同步 kickoff 放到线程里跑，绕开事件循环。纯 `.py` 脚本里用 `kickoff()` 没问题。

</div>

In [11]:
# Execute the crew's tasks
result = await deep_research_crew.kickoff_async(inputs=inputs)   # 🆕 原为 deep_research_crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Evaluate the top five emerging AI tools for automating competitive        │
│  market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing   │
│  firm." into specific topics and key questions that need investigation. Create a focused research plan.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_memory executed with result: Found memories:
- (score=0.80) The research plan focuses on evaluating the top five emerging AI tools for automating competitive market analysis.
  categories: AI Tools, Market Analysis, Research Plan...
Tool search_memory executed with result: Found memories:
- (score=0.73) Outmano is an AI-powered competitor tracker that automates the research process by monitoring competitors' pricing, product launches, SEO changes, and reviews, starting ...


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Research Plan: Evaluating Emerging AI Tools for Automating Competitive Market Analysis                     │
│                                                                                                                 │
│  #### 1. Main Research Topics to Investigate                                                                    │
│  1. **Identification of Top AI Tools**                                                                          │
│     - Focus: Outmano, Cohorva, MarketAgent, Agensio, and Rivalize.                                              │
│                                                                                                                 │
│  2. **Features and Functionalities**                                                                            │
│     - Examine key features offered by each tool, such as tracking capabilities, reporting, integration, and     │
│  automation functionalities.                                                                                    │
│                                                                                                                 │
│  3. **Limitations and Challenges**                                                                              │
│     - Investigate potential limitations each tool may have, such as effectiveness in specific industries and    │
│  data privacy concerns.                                                                                         │
│                                                                                                                 │
│  4. **Cost Analysis**                                                                                           │
│     - Analyze the pricing models of each tool, including subscription costs, potential hidden costs, and any    │
│  additional necessary expenses (training, maintenance).                                                         │
│                                                                                                                 │
│  5. **Ideal Use Cases**                                                                                         │
│     - Explore successful case studies or scenarios where these tools have been effectively implemented,         │
│  specifically tailored for mid-sized marketing firms.                                                           │
│                                                                                                                 │
│  #### 2. Key Questions for Each Topic                                                                           │
│  1. **Identification of Top AI Tools**                                                                          │
│     - What are the top five emerging AI tools currently available for automating competitive market analysis?   │
│     - What criteria were used to select these tools?                                                            │
│                                                                                                                 │
│  2. **Features and Functionalities**                                                                            │
│     - What unique features do each of the identified tools offer for market analysis?                           │
│     - How do these tools integrate with existing market

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, search the internet and scrape relevant websites to collect comprehensive       │
│  information on all identified topics. Verify information across multiple sources and cite all sources used.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query "Evaluate the top five emerging AI tools   │
│  for automating competitive market analysis, including their features, limitations, costs, and ideal use cases  │
│  for a mid-sized marketing firm." using all verified research data. Structure it with clear sections, include   │
│  citations, and provide actionable insights. ### START CODE HERE ### Use the custom tool to create intuitive    │
│  charts based on the verified information.  Include the generated charts into the final report, any chart       │
│  should be loadded from the deep_research_crew/plots/ path, so you just append the file name after that' ###    │
│  END CODE HERE ###                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool create_custom_plots executed with result: Successfully created 1 plots: plots/plot_1_20261021_230149.png...


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Report: Evaluating Emerging AI Tools for Automating Competitive Market Analysis                │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  This report evaluates five emerging AI tools for automating competitive market analysis: Outmano, Cohorva,     │
│  MarketAgent, Agensio, and Rivalize. The analysis covers each tool's features, limitations, costs, and ideal    │
│  use cases tailored for mid-sized marketing firms. The findings indicate that while each tool offers unique     │
│  functionalities and pricing structures, firms must consider their specific operational needs and budget        │
│  constraints in their selection process.                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Detailed Findings                                                                                           │
│                                                                                                                 │
│  ### 1. Overview of the Tools                                                                                   │
│                                                                                                                 │
│  #### **1.1 Outmano**                                                                                           │
│  - **Features**: Outmano tracks pricing, SEO, ads, content, features, hiring, and reviews. It provides weekly   │
│  digests and alerts via email and Slack, including automated change detection and sales battlecards.            │
│  - **Limitations**: Best suited for small to mid-sized firms; it may not handle large enterprise data           │
│  effectively.                                                                                                   │
│  - **Cost**:                                                                                                    │
│    - Starter: $29.40/month (up to 5 competitors)                                                                │
│    - Growth: $59.40/month (up to 10 competitors)                                                                │
│    - Pro: $119.40/month (up to 25 competitors)                                                                  │
│  - **Use Cases**: Ideal for early-stage startups seeking industry monitoring and pricing insights.              │
│  - **Sources**: [Outmano Pricing](https://outmano.com/pricing)                                                  │
│                                                                                                                 │
│  #### **1.2 Cohorva**                                                                                           │
│  - **Features**: Cohorva employs AI agents for real-time competitor analysis, pricing changes monitoring, and   │
│  content updates. It focuses significantly on automatio

Report successfully saved to: research_report-p2.md


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧪 看结果

</div>

In [12]:
import re
from IPython.display import display

report = Path("research_report-p2.md").read_text()
print("报告中的图片链接:", re.findall(r"!\[[^\]]*\]\(([^)]+)\)", report))   # 看看是不是 deep_research_crew/plots/...
print("实际生成的图:", sorted(Path("plots").glob("*.png"))[-3:] if Path("plots").exists() else "无")
display(Markdown(report))

报告中的图片链接: ['plots/plot_1_20261021_230149.png']
实际生成的图: [PosixPath('plots/plot_1_20261021_230149.png')]


# Comprehensive Report: Evaluating Emerging AI Tools for Automating Competitive Market Analysis

## Executive Summary
This report evaluates five emerging AI tools for automating competitive market analysis: Outmano, Cohorva, MarketAgent, Agensio, and Rivalize. The analysis covers each tool's features, limitations, costs, and ideal use cases tailored for mid-sized marketing firms. The findings indicate that while each tool offers unique functionalities and pricing structures, firms must consider their specific operational needs and budget constraints in their selection process.

---

## Detailed Findings

### 1. Overview of the Tools

#### **1.1 Outmano**
- **Features**: Outmano tracks pricing, SEO, ads, content, features, hiring, and reviews. It provides weekly digests and alerts via email and Slack, including automated change detection and sales battlecards.
- **Limitations**: Best suited for small to mid-sized firms; it may not handle large enterprise data effectively.
- **Cost**:
  - Starter: $29.40/month (up to 5 competitors)
  - Growth: $59.40/month (up to 10 competitors)
  - Pro: $119.40/month (up to 25 competitors)
- **Use Cases**: Ideal for early-stage startups seeking industry monitoring and pricing insights.
- **Sources**: [Outmano Pricing](https://outmano.com/pricing)

#### **1.2 Cohorva**
- **Features**: Cohorva employs AI agents for real-time competitor analysis, pricing changes monitoring, and content updates. It focuses significantly on automation.
- **Limitations**: Requires initial training for users to optimize its capabilities.
- **Cost**: Not explicitly detailed; aligns generally with market competitors.
- **Use Cases**: Suitable for marketing teams needing in-depth insights into competitor strategies.
- **Sources**: [Cohorva Competitor Analysis](https://cohorva.com/use-cases/competitor-analysis)

#### **1.3 MarketAgent**
- **Features**: MarketAgent offers self-service AI solutions for competitive analysis, allowing firms to track their market position through social media and reviews.
- **Limitations**: Its functionalities are limited for scaling in larger enterprises.
- **Cost**: Starting at $99/month.
- **Use Cases**: Targeted toward small firms to help them understand competitive strategies.
- **Sources**: [MarketAgent Blog](https://marketagent.ai/blog/competitive-intelligence-tools-for-small-business)

#### **1.4 Agensio**
- **Features**: Provides 24/7 monitoring across 50+ sources (competitor sites, press, LinkedIn, reviews), along with pricing detection and funding updates.
- **Limitations**: Primarily effective in French and European markets, which may limit applicability for international competitors.
- **Cost**:
  - Starter Pack: €82.50/month
  - Growth Pack: €195/month
  - Suite Package: €450/month
- **Use Cases**: Best suited for European small and medium-sized businesses (SMBs) needing continuous market intelligence.
- **Sources**: [Agensio Pricing](https://agensio.ai/en/pricing)

#### **1.5 Rivalize**
- **Features**: Rivalize focuses on monitoring competitor activity, providing real-time insights, change alerts, and proactive tracking of competitor pricing.
- **Limitations**: Primarily designed for smaller businesses, with advanced features possibly incurring extra costs.
- **Cost**:
  - Starter: $19/month (3 competitors)
  - Growth: $79/month (10 competitors)
  - Pro: $199/month (25 competitors)
- **Use Cases**: Effective for startups needing essential competitive insights.
- **Sources**: [Rivalize Pricing](https://rivalize.ai/pricing)

---

### 2. Supporting Evidence and Analysis

#### 2.1 Comparative Analysis of Features and Pricing

| Tool       | Features Overview                                                                                                     |
|------------|------------------------------------------------------------------------------------------------------------------------|
| Outmano    | Tracks pricing, SEO, ads, content, features, hiring, and reviews. Weekly digests and automated change detection.        |
| Cohorva    | Real-time competitor analysis, pricing changes monitoring, content updates via AI agents.                            |
| MarketAgent | Self-service AI tracking through social media and reviews for market position analysis.                              |
| Agensio    | 24/7 monitoring across 50+ data sources, with pricing detection and centralized dashboard.                          |
| Rivalize   | Comprehensive monitoring, providing strategic insights through real-time tracking of competitor activities.              |

| Tool       | Pricing (Per Month)                             |
|------------|-----------------------------------------------|
| Outmano    | Starter: $29.40, Growth: $59.40, Pro: $119.40   |
| Cohorva    | Not publicly available; competitive pricing expected.  |
| MarketAgent | Starting at $99/month                           |
| Agensio    | Starter: €82.50, Growth: €195, Suite: €450      |
| Rivalize    | Starter: $19, Growth: $79, Pro: $199             |

![Comparative Analysis of AI Tools for Market Analysis](plots/plot_1_20261021_230149.png)

---

### 3. Key Insights and Recommendations

- **Target Audience**: Outmano and Rivalize offer competitive, cost-effective solutions tailored primarily for small to mid-sized firms. In contrast, Agensio serves firms with European market interests.
- **Feature Utilization**: Firms should assess their specific competitive intelligence needs. For example, those requiring real-time automated reporting may prefer Cohorva or Rivalize, while firms needing comprehensive analysis might choose Agensio or MarketAgent.
- **Budget Considerations**: Pricing structures significantly influence adoption. With options starting as low as $19 for Rivalize, budget-conscious firms can efficiently explore competitive analysis tools.
- **Operational Suitability**: Identifying individual firm requirements along with specific market focus (regional vs. global) will aid in selecting the most appropriate tool.

---

### 4. Complete Source Citations
- Outmano. (n.d.). Pricing. Retrieved from [Outmano Pricing](https://outmano.com/pricing)
- Cohorva. (n.d.). Use Cases: Competitor Analysis. Retrieved from [Cohorva Competitor Analysis](https://cohorva.com/use-cases/competitor-analysis)
- MarketAgent. (n.d.). Blog on Competitive Intelligence Tools. Retrieved from [MarketAgent Blog](https://marketagent.ai/blog/competitive-intelligence-tools-for-small-business)
- Agensio. (n.d.). Pricing and Competitive Intelligence Agent. Retrieved from [Agensio Pricing](https://agensio.ai/en/pricing)
- Rivalize. (n.d.). Pricing. Retrieved from [Rivalize Pricing](https://rivalize.ai/pricing)

---

This comprehensive report serves as a guideline for mid-sized marketing firms in evaluating AI tools to automate their competitive market analysis effectively. By aligning feature offerings and budget considerations with operational needs, firms can navigate the evolving landscape of market intelligence tools efficiently.

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| 在 Solution/ 里读 `config/tasks.yaml` | `FileNotFoundError`；或改成 `../config/tasks.yaml` 读到练习版 | 解答版 yaml 就在 `Solution/tasks.yaml` |
| 报告里图片路径和保存位置不一致 | 图片坏链接 | 工具存 `plots/`，task 说明也要写 `plots/` |
| notebook 里用 `kickoff()` | `RuntimeError: ... running event loop` | `await crew.kickoff_async(...)` |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. 参考答案的 description 说了哪三件事？</b></summary>

工具作用（根据研究结果自动画图）、输入要求（核查过的信息）、传参方式（全部已验证信息作为一个字符串）。

</details>

<details><summary><b>2. 为什么画图工具只给 Report Writer？</b></summary>

它的 context 是 fact checker 核查过的数据；在更早的 agent 上画图会把未验证的数字可视化。

</details>

<details><summary><b>3. 解答版 tasks.yaml 有什么不适用于本 notebook 的地方？</b></summary>

它要求从 `deep_research_crew/plots/` 加载图片，而工具实际存到 cwd 下的 `plots/`；模型若照做，报告图片链接就会失效（本次运行模型用了工具返回的真实路径，侥幸没出问题）。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：**答案 = 一份写清楚的工具说明书 + 把工具交给对的 agent + 在 task 里写明怎么用结果**；路径这种细节要和工具实际行为对齐。

</div>